In [ ]:
with open('/content/nse.py', 'r') as f:
    print('--- nse.py ---')
    print(f.read())

with open('/content/one.py', 'r') as f:
    print('\n--- one.py ---')
    print(f.read())

--- nse.py ---
#!/usr/bin/env python3

import time
import json
import csv
import argparse
from datetime import datetime

import pandas as pd
import requests

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By


def sanitize_filename(name):
    return (
        name.replace(" ", "_")
        .replace(":", "_")
        .replace("/", "_")
        .replace("&", "and")
    )


def get_date_time():
    # Returns date and time formatted as YYYYMMDD_HHMMSS
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def get_all_indices():

    print("Downloading master index list...")

    session = requests.Session()

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/136.0.0.0 Safari/537.36"
        ),
        "Accept": "application/json,text/plain,*/*",
        "Accept-Language": "en-US,en;q=0.9",

In [ ]:
import os
import sys

# 1. Cleanly download and install Google Chrome stable and matching chromedriver
print("Downloading and installing stable Google Chrome...")
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get install -f -y -q
!rm google-chrome-stable_current_amd64.deb

# Install standard selenium package
print("Installing Selenium package...")
!pip install -q selenium

from google.colab import drive

# 2. Mount Google Drive
print("\nMounting Google Drive...")
drive.mount('/content/drive')

# 3. Define and create the output directory in Google Drive
output_dir = "/content/drive/MyDrive/NSEDATA-ALL-INDEX"
os.makedirs(output_dir, exist_ok=True)
print(f"Target directory ready: {output_dir}")

# 4. Adapt nse.py functionality to download files directly to the target directory
original_cwd = os.getcwd()
try:
    # Change working directory to Google Drive so all output CSVs are saved there directly
    os.chdir(output_dir)
    print(f"Changed working directory to Google Drive: {os.getcwd()}")

    # Import functions from nse.py
    sys.path.append('/content')
    import nse

    # Inject standard headless options into the webdriver inside nse.py
    original_download_nse_csv = nse.download_nse_csv

    def patched_download_nse_csv(symbols):
        from selenium.webdriver.chrome.options import Options
        from selenium import webdriver

        print("\nInitializing Headless Chrome...")
        options = Options()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')

        # Maintain explicit reference to the genuine Chrome class to avoid recursion loop
        real_chrome_class = webdriver.Chrome

        # Standard webdriver instantiation with the corrected options
        original_chrome = nse.webdriver.Chrome
        nse.webdriver.Chrome = lambda *args, **kwargs: real_chrome_class(options=options)
        try:
            original_download_nse_csv(symbols)
        finally:
            nse.webdriver.Chrome = original_chrome

    # Fetch all indices
    print("Fetching all indices...")
    indexes = nse.get_all_indices()
    print(f"Found {len(indexes)} indices to process.")

    # Download all indices using the patched method
    patched_download_nse_csv(indexes)

finally:
    # Restore original working directory
    os.chdir(original_cwd)
    print(f"Restored original working directory: {os.getcwd()}")

Selecting previously unselected package google-chrome-stable.
(Reading database ... 123406 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (157.0.8103.78-1) ...
dpkg: dependency problems prevent configuration of google-chrome-stable:
 google-chrome-stable depends on libatk-bridge2.0-0 (>= 2.5.3); however:
  Package libatk-bridge2.0-0 is not installed.
 google-chrome-stable depends on libatk1.0-0 (>= 2.11.90); however:
  Package libatk1.0-0 is not installed.
 google-chrome-stable depends on libatspi2.0-0 (>= 2.9.90); however:
  Package libatspi2.0-0 is not installed.
 google-chrome-stable depends on libxcomposite1 (>= 1:0.4.4-1); however:
  Package libxcomposite1 is not installed.

dpkg: error processing package google-chrome-stable (--install):
 dependency problems - leaving unconfigured
Processing triggers for man-db (2.12.0-4build2) ...
Errors were encountered while processing:
 google-chrome-st

In [ ]:
import sys
import os

# 1. Install Google Chrome Stable and dependencies on Google Colab
print("Downloading and installing stable Google Chrome...")
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get install -f -y -q
!rm google-chrome-stable_current_amd64.deb

print("Installing Selenium...")
!pip install -q selenium

# Recreate nse.py with robust options bypassing HTTP/2 issues
nse_code = r"""import time
import json
import csv
import argparse
from datetime import datetime
import pandas as pd
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

def sanitize_filename(name):
    return name.replace(" ", "_").replace(":", "_").replace("/", "_").replace("&", "and")

def get_date_time():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def get_all_indices():
    print("Downloading master index list...")
    session = requests.Session()
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept": "application/json,text/plain,*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.nseindia.com/",
    }
    session.get("https://www.nseindia.com/", headers=headers, timeout=30)
    response = session.get("https://www.nseindia.com/api/allIndices", headers=headers, timeout=30)
    response.raise_for_status()
    data = response.json()
    df = pd.DataFrame(data["data"])
    master_file = f"NSE_ALL_INDICES_{get_date_time()}.csv"
    df.to_csv(master_file, index=False)
    print(f"Saved: {master_file}")
    print(f"Rows : {len(df)}")
    if "index" not in df.columns:
        raise Exception(f"Column 'index' not found.")
    return df["index"].dropna().astype(str).str.strip().tolist()

def download_nse_csv(symbols):
    print("\nInitializing Chrome...")
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-http2')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36')

    # Let Selenium Manager find both the newly installed Chrome and its compatible driver automatically
    driver = webdriver.Chrome(options=options)
    try:
        for idx, index_symbol in enumerate(symbols, 1):
            print("="*80)
            print(f"[{idx}/{len(symbols)}] {index_symbol}")
            print("="*80)
            url_encoded = index_symbol.replace(" ", "%20").replace("&", "%26")
            url = f"https://www.nseindia.com/api/equity-stock-indices?index={url_encoded}"
            print(url)
            try:
                driver.get(url)
                time.sleep(2)
                pre = driver.find_element(By.TAG_NAME, "pre")
                data = json.loads(pre.text)
                if "data" not in data:
                    print("Invalid JSON returned by NSE.")
                    continue
                filename = f"{sanitize_filename(index_symbol)}.csv"
                with open(filename, "w", newline="", encoding="utf-8") as f:
                    writer = csv.writer(f)
                    writer.writerow(["symbol", "open", "high", "low", "ltP", "ptsC30d", "volume", "value_crores", "yHigh", "yLow", "pChange"])
                    for item in data["data"]:
                        writer.writerow([
                            item.get("symbol", "-"),
                            item.get("open", "-"),
                            item.get("high", "-"),
                            item.get("low", "-"),
                            item.get("ltP", "-"),
                            item.get("ptsC30d", "-"),
                            item.get("totalTradedVolume", "-"),
                            item.get("totalTradedValue", "-"),
                            item.get("yearHigh", "-"),
                            item.get("yearLow", "-"),
                            item.get("pChange", "-")
                        ])
                print(f"Saved: {filename}")
            except Exception as e:
                print(f"Failed: {index_symbol}")
                print(e)
    finally:
        driver.quit()
        print("\nChrome session closed.")
"""

with open('/content/nse.py', 'w') as f:
    f.write(nse_code.strip())

# Ensure the /content directory is at the absolute front of the import path
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# Reload nse module
if 'nse' in sys.modules:
    import importlib
    importlib.reload(sys.modules['nse'])
else:
    import nse

# Define target directory on Google Drive
output_dir = "/content/drive/MyDrive/NSEDATA-ALL-INDEX"
os.makedirs(output_dir, exist_ok=True)

original_cwd = os.getcwd()
try:
    os.chdir(output_dir)
    print("Fetching all indices...")
    indexes = nse.get_all_indices()
    print(f"Found {len(indexes)} indices to process.")
    # Run the downloads using our robust configuration
    nse.download_nse_csv(indexes)
finally:
    os.chdir(original_cwd)
    print(f"Restored original working directory: {os.getcwd()}")

Selecting previously unselected package google-chrome-stable.
(Reading database ... 123406 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (157.0.8103.78-1) ...
dpkg: dependency problems prevent configuration of google-chrome-stable:
 google-chrome-stable depends on libatk-bridge2.0-0 (>= 2.5.3); however:
  Package libatk-bridge2.0-0 is not installed.
 google-chrome-stable depends on libatk1.0-0 (>= 2.11.90); however:
  Package libatk1.0-0 is not installed.
 google-chrome-stable depends on libatspi2.0-0 (>= 2.9.90); however:
  Package libatspi2.0-0 is not installed.
 google-chrome-stable depends on libxcomposite1 (>= 1:0.4.4-1); however:
  Package libxcomposite1 is not installed.

dpkg: error processing package google-chrome-stable (--install):
 dependency problems - leaving unconfigured
Processing triggers for man-db (2.12.0-4build2) ...
Errors were encountered while processing:
 google-chrome-st

In [ ]:
import sys
import os
from datetime import datetime
import json
import csv
import time
import pandas as pd
import requests
from google.colab import drive

# 1. Install Google Chrome and Selenium packages
print("1. Setting up Chrome environment & Selenium packages...")
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get install -f -y -q
!rm google-chrome-stable_current_amd64.deb
!pip install -q selenium

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# 2. Check and Mount Google Drive
print("\n2. Verifying Google Drive mount status...")
if not os.path.exists('/content/drive/MyDrive'):
    try:
        drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print("Could not mount Google Drive via program. Ensure it is connected via the file manager panel:", e)
else:
    print("Google Drive is already mounted and active.")

# 3. Create target directory with Run Date (YYYYMMDD)
base_dir = "/content/drive/MyDrive/NSEDATA-ALL-INDEX"
run_date = datetime.now().strftime("%Y%m%d")  # Format: YYYYMMDD
output_dir = os.path.join(base_dir, run_date)
os.makedirs(output_dir, exist_ok=True)
print(f"All output files for today ({run_date}) will be saved in: {output_dir}")

# 4. Core Download Program
def sanitize_filename(name):
    return name.replace(" ", "_").replace(":", "_").replace("/", "_").replace("&", "and")

def get_date_time():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def get_all_indices():
    print("Downloading master index list...")
    session = requests.Session()
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept": "application/json,text/plain,*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.nseindia.com/",
    }
    session.get("https://www.nseindia.com/", headers=headers, timeout=30)
    response = session.get("https://www.nseindia.com/api/allIndices", headers=headers, timeout=30)
    response.raise_for_status()
    data = response.json()
    df = pd.DataFrame(data["data"])
    master_file = f"NSE_ALL_INDICES_{get_date_time()}.csv"
    df.to_csv(master_file, index=False)
    print(f"Master index saved: {master_file} ({len(df)} records)")
    return df["index"].dropna().astype(str).str.strip().tolist()

def download_nse_csv(symbols):
    print("\nInitializing Headless Chrome with HTTP/2 bypassed...")
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-http2')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36')

    driver = webdriver.Chrome(options=options)
    try:
        for idx, index_symbol in enumerate(symbols, 1):
            print("="*80)
            print(f"[{idx}/{len(symbols)}] Downloading {index_symbol}")
            print("="*80)
            # Clean parameter encoding
            url_encoded = index_symbol.replace(" ", "%20").replace("&", "%26")
            url = f"https://www.nseindia.com/api/equity-stock-indices?index={url_encoded}"

            try:
                driver.get(url)
                time.sleep(2.5)
                pre = driver.find_element(By.TAG_NAME, "pre")
                data = json.loads(pre.text)
                if "data" not in data:
                    print("Warning: Invalid payload returned by NSE.")
                    continue

                filename = f"{sanitize_filename(index_symbol)}.csv"
                with open(filename, "w", newline="", encoding="utf-8") as f:
                    writer = csv.writer(f)
                    writer.writerow(["symbol", "open", "high", "low", "ltP", "ptsC30d", "volume", "value_crores", "yHigh", "yLow", "pChange"])
                    for item in data["data"]:
                        writer.writerow([
                            item.get("symbol", "-"),
                            item.get("open", "-"),
                            item.get("high", "-"),
                            item.get("low", "-"),
                            item.get("ltP", "-"),
                            item.get("ptsC30d", "-"),
                            item.get("totalTradedVolume", "-"),
                            item.get("totalTradedValue", "-"),
                            item.get("yearHigh", "-"),
                            item.get("yearLow", "-"),
                            item.get("pChange", "-")
                        ])
                print(f"Successfully saved data: {filename}")
            except Exception as e:
                print(f"Error processing {index_symbol}: {e}")
    finally:
        driver.quit()
        print("\nChrome browser session terminated.")

# 5. Run Scraper
original_cwd = os.getcwd()
try:
    os.chdir(output_dir)
    indexes = get_all_indices()
    print(f"Initiating automated download for {len(indexes)} indices...")
    download_nse_csv(indexes)
finally:
    os.chdir(original_cwd)
    print(f"Restored environment workspace: {os.getcwd()}")

1. Setting up Chrome environment & Selenium packages...
(Reading database ... 123788 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (157.0.8103.78-1) over (157.0.8103.78-1) ...
Setting up google-chrome-stable (157.0.8103.78-1) ...
Processing triggers for man-db (2.12.0-4build2) ...

2. Verifying Google Drive mount status...
Google Drive is already mounted and active.
All output files for today (20261114) will be saved in: /content/drive/MyDrive/NSEDATA-ALL-INDEX/20261114
Master index saved: NSE_ALL_INDICES_20261114_161118.csv (139 records)
Initiating automated download for 139 indices...

Initializing Headless Chrome with HTTP/2 bypassed...
[1/139] Downloading NIFTY 50
Successfully saved data: NIFTY_50.csv
[2/139] Downloading NIFTY NEXT 50
Successfully saved data: NIFTY_NEXT_50.csv
[3/139] Downloading NIFTY BANK
Successfully saved data: NIFTY_BANK.csv
[4/139] Downloading NIFTY FINANCIAL SERVICES


In [ ]:
import pandas as pd
import os
from datetime import datetime
import time

# Dynamic folder search
run_date = datetime.now().strftime("%Y%m%d")
target_dir = f'/content/drive/MyDrive/NSEDATA-ALL-INDEX/{run_date}'
file_path = os.path.join(target_dir, 'NIFTY_50.csv')

print(f"Target path: {file_path}")

# Check if directory exists first
if os.path.exists(target_dir):
    # Wait up to 5 seconds for Drive sync to stabilize if needed
    for _ in range(5):
        if os.path.exists(file_path):
            break
        time.sleep(1)

    # List directory contents to see what was saved
    files = os.listdir(target_dir)
    print(f"\nSuccess: Folder found! Total files in folder: {len(files)}")
    print("Sample of files found:", files[:5])

    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"\nFirst 5 rows of NIFTY_50.csv:")
        display(df.head(5))
    else:
        print(f"\nNIFTY_50.csv not found in folder. Check spelling or if download skipped it.")
else:
    print(f"\nError: Directory {target_dir} does not exist yet. Please run cell dc0c7785 first.")

Target path: /content/drive/MyDrive/NSEDATA-ALL-INDEX/20261114/NIFTY_50.csv

Success: Folder found! Total files in folder: 138
Sample of files found: ['NIFTY100_ALPHA_30.csv', 'NIFTY500_MULTICAP_INDIA_MANUFACTURING_50_30_20.csv', 'NIFTY_TOTAL_MARKET.csv', 'NIFTY_MIDSMALL_HEALTHCARE.csv', 'NIFTY200_QUALITY_30.csv']

First 5 rows of NIFTY_50.csv:


,symbol,open,high,low,ltP,ptsC30d,volume,value_crores,yHigh,yLow,pChange
0,NIFTY 50,25209.85,-,-,-,-,239573267,2.539710e+11,26373.2,22179.9,-0.61
1,HDFCBANK,808.00,-,-,-,-,23442886,1.879135e+10,1017.5,669.2,-1.02
2,RELIANCE,1336.60,-,-,-,-,13756247,1.840710e+10,1611.8,1152.2,0.00
3,ICICIBANK,1576.10,-,-,-,-,11440109,1.801142e+10,1616.0,1187.6,-0.40
4,SBIN,1090.00,-,-,-,-,15886106,1.717383e+10,1234.7,933.9,-0.60


In [ ]:
import os
import shutil
from google.colab import drive

print("1. Properly disconnecting and unmounting Drive...")
try:
    drive.flush_and_unmount()
except Exception as e:
    print("Note during unmount:", e)

# Clean up any leftover empty mountpoint paths to avoid the 'already contains files' error
if os.path.exists('/content/drive'):
    try:
        shutil.rmtree('/content/drive')
    except Exception as e:
        print("Removing leftover mount directory failed, continuing:", e)

print("\n2. Remounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# 3. Force the OS to re-scan the target directory
base_path = '/content/drive/MyDrive/NSEDATA-ALL-INDEX'
if os.path.exists(base_path):
    # Accessing listdir triggers Google Drive sync for this folder
    contents = os.listdir(base_path)
    print(f"\nSuccess! Base path '{base_path}' successfully refreshed.")
    print(f"Visible directories and files: {contents}")
else:
    print(f"\nError: Base directory '{base_path}' not found. Please verify Google Drive connection.")

1. Properly disconnecting and unmounting Drive...
Drive not mounted, so nothing to flush and unmount.

2. Remounting Google Drive...
Mounted at /content/drive

Success! Base path '/content/drive/MyDrive/NSEDATA-ALL-INDEX' successfully refreshed.
Visible directories and files: ['NSE_ALL_INDICES_20261113_113945.csv', 'NSE_ALL_INDICES_20261113_114242.csv', 'NSE_ALL_INDICES_20261113_114306.csv', 'NSE_ALL_INDICES_20261113_114327.csv', 'NSE_ALL_INDICES_20261113_141607.csv', 'NSE_ALL_INDICES_20261113_141637.csv', 'NSE_ALL_INDICES_20261113_141920.csv']


In [ ]:
import os

target_subfolder = '/content/drive/MyDrive/NSEDATA-ALL-INDEX/20261114'

if os.path.exists(target_subfolder):
    files = sorted(os.listdir(target_subfolder))
    print(f"\nSuccess! Found the subfolder: {target_subfolder}")
    print(f"Total index files downloaded inside this folder: {len(files)}")
    print("\nFirst 20 index files:")
    for f in files[:20]:
        print(f" - {f}")
else:
    print(f"\nCould not find directory: {target_subfolder}")


Could not find directory: /content/drive/MyDrive/NSEDATA-ALL-INDEX/20261114
